# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/720-hz/flyrank-ml-internship/blob/main/work/notebooks/Week%204/w04_signal_audit.ipynb?flush_cache=true)

Three candidate signals, tested against the real data rather than assumed — the same tests
`work/notebooks/Week 4/w04_baseline_score.ipynb` and the Week 8 capstone paper cite directly.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"Rows: {len(df):,} | base rate (declining): {df['is_declining_label'].mean():.3f}")

Working dir: /home/claude/flyrank-ml-internship


Rows: 30,000 | base rate (declining): 0.542


## 1. Distributions

*Look before deciding: distributions of key fields. Note the heavy tails.*

`impressions_90d` and `clicks_90d` are classic heavy-tailed web-traffic distributions — mean far above
median, driven by a small number of very high-traffic pages. `ctr` is stored as a percentage
(e.g. `0.76` = 0.76%) and is similarly right-skewed with a long tail of near-zero values.

In [2]:
for col in ["impressions_90d", "clicks_90d", "ctr", "avg_position", "content_age_days", "days_since_last_update"]:
    s = df[col].dropna()
    print(f"{col:<24} n={len(s):>6,}  mean={s.mean():>10.2f}  median={s.median():>9.2f}  "
          f"p95={s.quantile(0.95):>10.2f}  max={s.max():>10.2f}")

print(f"\nword_count missing: {df['word_count'].isna().sum():,} of {len(df):,} rows ({df['word_count'].isna().mean():.1%})"
      f" -- flagged as missingness, never silently filled, in the safe feature set (ML-05).")

impressions_90d          n=30,000  mean=   5200.37  median=   731.00  p95=  22996.50  max= 517715.00
clicks_90d               n=30,000  mean=     16.10  median=     1.00  p95=     69.05  max=   4178.00
ctr                      n=30,000  mean=      0.51  median=     0.07  p95=      1.09  max=    100.00
avg_position             n=30,000  mean=     16.34  median=    10.80  p95=     48.20  max=    245.00
content_age_days         n=30,000  mean=    256.17  median=   236.00  p95=    487.00  max=    564.00
days_since_last_update   n=30,000  mean=     46.10  median=    20.00  p95=    104.00  max=    373.00

word_count missing: 7,699 of 30,000 rows (25.7%) -- flagged as missingness, never silently filled, in the safe feature set (ML-05).


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Signal 1 — staleness, behind FlyRank's refresh-tier flags.** Assumption: pages get more likely to be
declining the longer it's been since their last update.

In [3]:
bins = [-1, 30, 90, 180, 365, np.inf]
tier_labels = ["0-30", "31-90", "91-180", "181-365", "365+"]
df["freshness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=tier_labels)

staleness_table = (
    df.groupby("freshness_bucket", observed=True)["is_declining_label"]
    .agg(n="count", decline_rate="mean")
    .reindex(tier_labels)
)
print("Signal 1 -- staleness vs. decline rate")
print(staleness_table.round(3))
print("\nVerdict: MIXED -- decline rate rises through 91-180, then drops at 181-365; the two oldest"
      " buckets (31-90, 365+) barely have enough pages (n=175, n=5) to trust.")

Signal 1 -- staleness vs. decline rate
                      n  decline_rate
freshness_bucket                     
0-30              20480         0.511
31-90               175         0.589
91-180             9171         0.611
181-365             169         0.467
365+                  5         0.600

Verdict: MIXED -- decline rate rises through 91-180, then drops at 181-365; the two oldest buckets (31-90, 365+) barely have enough pages (n=175, n=5) to trust.


**Signal 2 — CTR vs. position, behind FlyRank's `needs_ctr_fix` flag.** Assumption: pages ranking well
but getting fewer clicks than their position should earn are worth reviewing.

In [4]:
pos_bins = [0, 3, 10, 20, 50, np.inf]
pos_labels = ["top_3", "page_1", "striking", "page_3_5", "deep"]
has_pos = df["avg_position"] > 0
df.loc[has_pos, "position_tier"] = pd.cut(df.loc[has_pos, "avg_position"], bins=pos_bins, labels=pos_labels)

ctr_table = (
    df[has_pos]
    .groupby("position_tier", observed=True)
    .agg(n=("ctr", "count"), mean_ctr=("ctr", "mean"), decline_rate=("is_declining_label", "mean"))
    .reindex(pos_labels)
)
print(f"Signal 2 -- CTR by position tier (n with real position data: {has_pos.sum():,} / {len(df):,})")
print(ctr_table.round(3))
print("\nVerdict: CONFIRMED -- mean CTR falls cleanly from top_3 to deep, with solid n at every tier.")

Signal 2 -- CTR by position tier (n with real position data: 28,795 / 30,000)
                   n  mean_ctr  decline_rate
position_tier                               
top_3           1141     2.714         0.498
page_1         11842     0.651         0.569
striking        7273     0.323         0.610
page_3_5        7225     0.222         0.562
deep            1314     0.151         0.343

Verdict: CONFIRMED -- mean CTR falls cleanly from top_3 to deep, with solid n at every tier.


**Signal 3 — thin content, behind FlyRank's `thin_visible_page` flag** (`word_count > 0`,
`word_count < 1200`, `impressions_90d >= 250`). Assumption: shorter pages are more likely to be declining.

In [5]:
tier_order = ["<1000", "1000-2000", "2000-3500", "3500+"]
wc_table = (
    df.groupby("word_count_tier", observed=True)["is_declining_label"]
    .agg(n="count", decline_rate="mean")
    .reindex(tier_order)
)
print("Signal 3 -- word_count tier vs. decline rate")
print(wc_table.round(3))
print(f"\n(missing word_count_tier: n={df['word_count_tier'].isna().sum():,}, "
      f"decline_rate={df.loc[df['word_count_tier'].isna(), 'is_declining_label'].mean():.3f})")
print(f"overall base rate: {df['is_declining_label'].mean():.3f}")
print("\nVerdict: OPPOSITE -- the shortest tier (<1000 words) has the LOWEST decline rate (0.207), not the"
      " highest, directly contradicting the 'thin content declines more' assumption behind thin_visible_page.")

Signal 3 -- word_count tier vs. decline rate
                     n  decline_rate
word_count_tier                     
<1000              973         0.207
1000-2000         3780         0.556
2000-3500        11263         0.588
3500+             6285         0.597

(missing word_count_tier: n=7,699, decline_rate=0.465)
overall base rate: 0.542

Verdict: OPPOSITE -- the shortest tier (<1000 words) has the LOWEST decline rate (0.207), not the highest, directly contradicting the 'thin content declines more' assumption behind thin_visible_page.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Picking **Signal 2**, since it is the one an actual FlyRank product flag (`needs_ctr_fix`) is built on.
The assumption behind that flag — CTR falls as position gets worse — holds up cleanly in this data
(Signal 2, verdict CONFIRMED): mean CTR drops in a straight line from the top-3 tier down to deep
positions, with solid sample size (well over a thousand pages) at every tier. This is the one signal
of the three that a content team could trust as-is without a caveat, and it's the signal the baseline
rule (`work/notebooks/Week 4/w04_baseline_score.ipynb`) is ultimately built on.

## 4. What this means in practice

Only one of three plausible, flag-linked assumptions survives contact with the data unqualified: CTR-vs-position.
Staleness is a MIXED signal — "this page is old, refresh it" is not a supported standalone rule here, and it
never appears as one in the baseline score or the capstone's recommendation queue. Thin content is the most
surprising result — it's the OPPOSITE of the assumption behind `thin_visible_page`, which is a reminder that a
plausible-sounding content-quality rule still needs a real test before a reviewer trusts it. A content team
should treat CTR-vs-position as the one safe, ready-to-use signal from this audit, and treat both staleness
and word count as inputs that need a human's judgment call, not an automatic trigger.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.